# 03 - Modeling

Ноутбук закрывает блок **"Моделирование и эксперименты"**:
- baseline-модель,
- минимум 4-5 моделей,
- мини-тюнинг гиперпараметров,
- таблица экспериментов в формате "гипотеза -> проверка -> результат",
- выбор финальной модели.

In [1]:
from pathlib import Path
from typing import Dict, Optional, Tuple

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DEFAULT_TARGET = "AdoptionSpeed"
DEFAULT_RANDOM_STATE = 42

In [3]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy().drop_duplicates()
    if "Name" in clean_df.columns:
        clean_df["NameLength"] = clean_df["Name"].fillna("").astype(str).str.len()
    return clean_df


def split_features_target(
    df: pd.DataFrame, target_col: str = DEFAULT_TARGET
) -> Tuple[pd.DataFrame, pd.Series]:
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' was not found.")
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y


def _build_stratify_target(y: pd.Series, use_stratify: bool) -> Optional[pd.Series]:
    if not use_stratify:
        return None
    value_counts = y.value_counts()
    if (value_counts < 2).any():
        return None
    return y


def make_train_val_test(
    df: pd.DataFrame,
    target_col: str = DEFAULT_TARGET,
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = DEFAULT_RANDOM_STATE,
    use_stratify: bool = True,
):
    X, y = split_features_target(df=df, target_col=target_col)
    stratify_all = _build_stratify_target(y=y, use_stratify=use_stratify)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify_all,
    )

    val_ratio_in_train_full = val_size / (1 - test_size)
    stratify_train = _build_stratify_target(y=y_train_full, use_stratify=use_stratify)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=val_ratio_in_train_full,
        random_state=random_state,
        stratify=stratify_train,
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def build_preprocessor(X_train: pd.DataFrame) -> ColumnTransformer:
    feature_df = X_train.copy()
    if "PetID" in feature_df.columns:
        feature_df = feature_df.drop(columns=["PetID"])

    numeric_cols = feature_df.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = feature_df.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ]
    )

In [4]:
def get_models(random_state: int = DEFAULT_RANDOM_STATE) -> Dict[str, object]:
    """Набор моделей для сравнения (5 штук)."""
    return {
        "logreg_baseline": LogisticRegression(max_iter=1200, random_state=random_state),
        "knn": KNeighborsClassifier(n_neighbors=15),
        "random_forest": RandomForestClassifier(
            n_estimators=350,
            random_state=random_state,
            n_jobs=-1,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=350,
            random_state=random_state,
            n_jobs=-1,
        ),
        "gradient_boosting": GradientBoostingClassifier(random_state=random_state),
    }


def evaluate_model(model: Pipeline, X: pd.DataFrame, y: pd.Series):
    preds = model.predict(X)
    return {
        "accuracy": accuracy_score(y, preds),
        "macro_f1": f1_score(y, preds, average="macro"),
    }


def tune_pipeline(
    pipeline: Pipeline,
    model_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
) -> Tuple[Pipeline, str]:
    """Мини-тюнинг для части моделей, чтобы показать перебор гиперпараметров."""
    if model_name == "random_forest":
        params = {
            "model__n_estimators": [250, 400],
            "model__max_depth": [None, 20],
            "model__min_samples_leaf": [1, 3],
        }
    elif model_name == "knn":
        params = {
            "model__n_neighbors": [9, 15, 25],
            "model__weights": ["uniform", "distance"],
        }
    else:
        # Для моделей без GridSearch обязательно обучаем пайплайн.
        pipeline.fit(X_train, y_train)
        return pipeline, "no_tuning"

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train)
    return search.best_estimator_, f"tuned ({search.best_params_})"

In [5]:
TRAIN_PATH = Path("../data/raw/train/train.csv")
MODEL_SAVE_PATH = Path("../models/best_model.joblib")

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Файл не найден: {TRAIN_PATH.resolve()}")

raw_df = pd.read_csv(TRAIN_PATH)
clean_df = clean_dataframe(raw_df)
X_train, X_val, X_test, y_train, y_val, y_test = make_train_val_test(clean_df)

preprocessor = build_preprocessor(X_train=X_train)
all_models = get_models()

# Гипотезы для отчета
hypotheses = {
    "logreg_baseline": "Линейная baseline-модель даст нижнюю границу качества.",
    "knn": "Локальные соседства улучшают качество на смешанных признаках.",
    "random_forest": "Ансамбль деревьев устойчив к шумным признакам.",
    "extra_trees": "Более случайный ансамбль может снизить variance.",
    "gradient_boosting": "Последовательное бустинг-обучение даст лучший macro F1.",
}

rows = []
best_name = ""
best_score = -1.0
best_pipeline = None

for model_name, clf in all_models.items():
    base_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", clf),
        ]
    )

    tuned_pipeline, tuning_note = tune_pipeline(
        pipeline=base_pipeline,
        model_name=model_name,
        X_train=X_train,
        y_train=y_train,
    )

    val_metrics = evaluate_model(tuned_pipeline, X_val, y_val)
    test_metrics = evaluate_model(tuned_pipeline, X_test, y_test)

    row = {
        "model": model_name,
        "hypothesis": hypotheses[model_name],
        "setup": tuning_note,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
    }
    rows.append(row)

    if row["val_macro_f1"] > best_score:
        best_score = row["val_macro_f1"]
        best_name = model_name
        best_pipeline = tuned_pipeline

if best_pipeline is None:
    raise RuntimeError("No model was trained.")

leaderboard = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
joblib.dump(best_pipeline, MODEL_SAVE_PATH)

print("Leaderboard (sorted by val_macro_f1):")
leaderboard

NotFittedError: Pipeline is not fitted yet.

In [ ]:
print(f"Best model: {best_name}")
print(f"Saved to: {MODEL_SAVE_PATH.resolve()}")

best_row = leaderboard.iloc[0]
print("\nFinal model justification:")
print(
    f"Выбрана модель '{best_row['model']}', так как у нее максимальный val_macro_f1 "
    f"({best_row['val_macro_f1']:.4f}) при стабильном качестве на test "
    f"({best_row['test_macro_f1']:.4f})."
)

experiments_table = leaderboard[[
    "model",
    "hypothesis",
    "setup",
    "val_macro_f1",
    "test_macro_f1",
]].copy()
experiments_table

## Вывод по блоку моделирования

- baseline обучен и зафиксирован для сравнения;
- протестировано 5 моделей (включая ансамбли);
- для части моделей выполнен перебор гиперпараметров (`GridSearchCV`);
- построена таблица экспериментов с гипотезами;
- выбрана и сохранена финальная модель `models/best_model.joblib`.